# 3-8 groupby分组聚合

`groupby()` 按字段把数据拆成多个组，再对每组计算并合并结果，可以理解为“分组 → 计算 → 汇总”。

> 约定：使用 `pd` 作为 Pandas 的别名，使用 `np` 作为 NumPy 的别名。本节可独立从上到下运行。

In [ ]:
import pandas as pd
import numpy as np

## 1. 准备数据路径

本节读取本章 `data` 文件夹中的模拟练习数据，使用 `pathlib` 处理相对路径。

In [ ]:
from pathlib import Path

# 从本章目录运行时使用 data；从项目根目录运行时定位到本章。
data_dir = Path("data")
if not data_dir.is_dir():
    data_dir = Path("第3章-Pandas") / "data"
if not data_dir.is_dir():
    raise FileNotFoundError("未找到本章的 data 文件夹。")

data_dir.as_posix()

In [ ]:
data = pd.read_csv(data_dir / "groupby.csv")
data

## 2. 查看分组

`groupby()` 返回一个分组对象，此时还没有得到最终统计表。

In [ ]:
g = data.groupby("company")
g

将分组对象转成列表，可以看到每个“组名 + 子表”的组合。

In [ ]:
list(g)

使用 `get_group()` 取出指定组；下例取得公司 B 的全部记录。

In [ ]:
g.get_group("B")

## 3. 使用 agg 聚合

| 聚合方式 | 含义 |
|---|---|
| `min` / `max` | 最小值 / 最大值 |
| `sum` / `mean` / `median` | 总和 / 均值 / 中位数 |
| `count` | 非缺失值数量 |
| `size` | 组内行数，包含缺失值所在行 |
| `nunique` | 不同值的个数，默认不计缺失值 |

### 3.1 对数值列使用同一规则

明确选取年龄和薪水，避免混入其他文本列。

In [ ]:
data.groupby("company")[["age", "salary"]].agg("mean")

### 3.2 不同字段使用不同规则

字典的键是列名，值是该列的聚合方式。

In [ ]:
data.groupby("company").agg({"age": "mean", "salary": "median"})

### 3.3 同一列使用多个规则

统计薪水的中位数、均值和有效数量时，传入函数名称列表。

In [ ]:
data.groupby("company").agg({"age": "mean", "salary": ["median", "mean", "count"]})

结果列名会形成多层索引。若希望直接得到清晰的一层列名，可以使用命名聚合。

In [ ]:
data.groupby("company", as_index=False).agg(
    平均年龄=("age", "mean"),
    薪水中位数=("salary", "median"),
    平均薪水=("salary", "mean"),
    总人数=("salary", "size")
)

`as_index=False` 把公司保留为普通列，便于后续保存或连接其他表。

## 4. 区分 count、size 与 nunique

下面的数据包含缺失数值和重复姓名。

In [ ]:
df = pd.DataFrame({
    "Name": ["Alice", "Bob", "Mallory", "Mallory", "Bob", "Mallory"],
    "City": ["Seattle", "Seattle", "Portland", "Seattle", "Seattle", "Portland"],
    "Val": [4, 3, 3, np.nan, np.nan, 4]
})
df

In [ ]:
df.groupby("City").agg({"Val": ["count", "size"], "Name": "nunique"})

Seattle 共 4 行，所以 `size=4`；其中只有 2 个非空 `Val`，所以 `count=2`；不同姓名有 Alice、Bob、Mallory 三个，所以 `nunique=3`。

分组键本身为空时，`groupby()` 默认忽略这些行；需要保留该组时可设置 `dropna=False`。

## 5. 要点总结

- `groupby()` 建立分组，`get_group()` 查看一组，`agg()` 得到统计结果。
- 聚合字典可分别指定各列的规则，列表可让一列计算多个指标。
- `count` 数非缺失值，`size` 数行，`nunique` 数不同值。
- 命名聚合可直接生成清晰的结果列名。